# Knowledge graph using Neo 4J

## Understanding Indian bail judgements

> Data source from Kaggle  
> DOI: **willian oliveira. (2026). Indian-Bail-Judgments [Dataset]. Kaggle. https://doi.org/10.34740/KAGGLE/DSV/19614939**

In [1]:
import pandas
import re
import uuid
from dotenv import load_dotenv
import os
from neo4j import GraphDatabase, Session, Record, ManagedTransaction
from neo4j.exceptions import GqlError, CypherSyntaxError
import atexit
from IPython import get_ipython
from typing import Any
import json


from langchain_neo4j import Neo4jGraph
from langchain_core.tools import tool, ToolException
from langchain_ollama import ChatOllama
from langchain_classic.agents import (
    AgentExecutor,
    create_tool_calling_agent,
    tool,
)
from langchain_core.prompts import (ChatPromptTemplate,
    FewShotChatMessagePromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_core.messages import AIMessage, ToolMessage
from langchain_core.prompts import MessagesPlaceholder

In [2]:
# Load from environment
load_dotenv('.env', override=True)
NEO4J_URI = os.getenv('KG_DATABASE_URL')
NEO4J_USERNAME = os.getenv('KG_DATABASE_USER')
NEO4J_PASSWORD = os.getenv('KG_DATABASE_PASSWORD')
NEO4J_DATABASE = os.getenv('KG_DATABASE') or 'neo4j'
LLM_HOST = os.getenv('OLLAMA_HOST')
LLM_MODEL = os.getenv('OLLAMA_MODEL')

In [3]:
NEO4J_AUTH = (NEO4J_USERNAME, NEO4J_PASSWORD)

# 1. Initiate session
driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
session = driver.session(database=NEO4J_DATABASE)

# 2. Define the unified cleanup logic
def close_neo4j_driver():
  global driver
  global session
  if session is not None:
    print("\n[Neo4j Cleanup] Closing active session...")
    try:
      session.close()
      print("[Neo4j Cleanup] session closed successfully.")
    except Exception as e:
      print(f"[Neo4j Cleanup] Error closing session: {e}")
    finally:
      session = None
  if driver is not None:
    print("\n[Neo4j Cleanup] Closing active connection pool...")
    try:
      driver.close()
      print("[Neo4j Cleanup] Driver shut down successfully.")
    except Exception as e:
      print(f"[Neo4j Cleanup] Error closing driver: {e}")
    finally:
      driver = None


# 3. Hook 1: Handle standard Python exit (Kernel > Shut Down / Restart)
atexit.register(close_neo4j_driver)

# 4. Hook 2: Jupyter/IPython explicit magic hook (Optional extra layer)
ip = get_ipython()
if ip is not None:

  def ipython_cleanup_hook(ipython):
    close_neo4j_driver()

  # Fires right before the kernel wraps up its event loop
  ip.events.register("post_execute", lambda: None)  # Dummy to ensure registration context

In [4]:
# Neo 4J graph
kg = Neo4jGraph(
    url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD, database=NEO4J_DATABASE
)

# Schema of the knowledge graph
# schema = kg.get_schema
schema = """Node properties:\nCase {bias_flag: BOOLEAN, parity_argument_used: BOOLEAN, region: STRING, facts: STRING, legal_issues: STRING, judgment_reason: STRING, summary: STRING, bail_cancellation_case: BOOLEAN, landmark_case: BOOLEAN, crime_type: STRING, id: STRING, title: STRING, date: STRING}\nPerson {id: STRING, type: STRING, gender: STRING, prior_cases: STRING, name: STRING}\nBail {outcome_labellled: STRING, id: STRING, type: STRING, outcome: STRING}\nCourt {name: STRING}\nRelationship properties:\n\nThe relationships:\n(:Person)-[:ACCUSED_IN]->(:Case)\n(:Person)-[:JUDGED]->(:Case)\n(:Bail)-[:RELATED_TO]->(:Case)\n(:Court)-[:HEARD]->(:Case)"""
schema

'Node properties:\nCase {bias_flag: BOOLEAN, parity_argument_used: BOOLEAN, region: STRING, facts: STRING, legal_issues: STRING, judgment_reason: STRING, summary: STRING, bail_cancellation_case: BOOLEAN, landmark_case: BOOLEAN, crime_type: STRING, id: STRING, title: STRING, date: STRING}\nPerson {id: STRING, type: STRING, gender: STRING, prior_cases: STRING, name: STRING}\nBail {outcome_labellled: STRING, id: STRING, type: STRING, outcome: STRING}\nCourt {name: STRING}\nRelationship properties:\n\nThe relationships:\n(:Person)-[:ACCUSED_IN]->(:Case)\n(:Person)-[:JUDGED]->(:Case)\n(:Bail)-[:RELATED_TO]->(:Case)\n(:Court)-[:HEARD]->(:Case)'

In [5]:
# Creating a tool to execute Cypher queries against the Neo4j database

def _handle_error(error: ToolException) -> str:
    if isinstance(error, (GqlError, CypherSyntaxError)) :
        return f"""
{
    "error": "Invalid query syntax: {error.args[0]}. Check provided query's syntax and try again.",
    "retry": true
}
"""
    return f"Unhandled exception occured, {error}"


@tool
def execute_cypher_query(query: str) -> list[dict[str, Any]]:
    """
    Execute a Cypher query against the Neo4j database and return the results as a list of dictionaries.

    Parameters:
    - query (str): Actual Cypher query to be executed, should be a valid Cypher statement.

    Returns:
    - list[dict[str, Any]]: A list of dictionaries where each dictionary represents a record from the query result. Each key in the dictionary corresponds to a field in the record.
    """
    print(f"Attempting Query: {query}")
    result = kg.query(query)
    return_val = json.dumps(result)
    print(f"Result: {return_val}")
    return return_val

In [6]:
# Creating prompt template for the LLM to generate Cypher queries based on user questions

SYSTEM_PROMPT_TEMPLATE = """You are an expert Neo4j graph database engineer.
Your job is to read a dictionary with key value pairs and execute the exact Cypher statement(s) required to create or update the graph.

GRAPH SCHEMA:
{schema}

CRITICAL RULES:
1. Always use `MERGE` for nodes that are unique like Case, Court etc. while use `CREATE` for nodes that are not unique like Person, Bail etc.
2. Use unique identifying properties inside the `MERGE` clause, and set other properties using `ON CREATE SET` or `SET`.
3. Infer order of operations based on the relationships between nodes. For example, if a Person is related to a Case, ensure the Case node is created or merged before creating the Person node and establishing the relationship. Person nodes will require Case node's unique identifier to establish the relationship.
4. If a field contains comma- or semicolon-separated values (like tags, categories, or skills), use `split()` and `UNWIND` to create individual nodes for each item.
5. Invoke the Cypher execution tool directly with the generated query.  Note: Do not include any explanations or apologies in your responses. Do not respond to any questions that might ask anything else than 
for you to construct a Cypher statement. Do not include any text except the generated Cypher statement.
6. String escape the user inputs within input data to ensure Cypher syntax is not broken.
7. Keep one query for one tool call, don't try to merge multiple queries into a single tool call, use scratchpad for history or queries.
"""
tool_call_id_1 = str(uuid.uuid4())
tool_call_id_2 = str(uuid.uuid4())
tool_call_id_3 = str(uuid.uuid4())
tool_call_id_4 = str(uuid.uuid4())
tool_call_id_5 = str(uuid.uuid4())
examples = [
    {
        "input_data": json.dumps(
            {
            "case_id": "DEL-HC-2022-7819",
            "case_title": "State of NCT of Delhi vs. Manpreet Singh & Ors.",
            "court": "Delhi High Court",
            "date": "2022-09-14",
            "judge": "Justice Mukta Gupta, Justice Anish Dayal",
            "bail_type": "Regular",
            "bail_cancellation_case": True,
            "landmark_case": False,
            "accused_name": "Manpreet Singh, Harvinder Gill and others",
            "accused_gender": "Male",
            "prior_cases": "Yes",
            "bail_outcome": "Rejected",
            "bail_outcome_label_detailed": "Bail previously granted was revoked due to suppression of prior criminal record and active witness tampering.",
            "crime_type": "Armed Robbery & Criminal Conspiracy",
            "facts": "The incident involved the armed interception of a logistics van transporting cash. Bail was initially granted by the Sessions Court on medical grounds and parity. The State filed for cancellation after ballistic reports confirmed licensed weapon misuse and the complainant reported physical intimidation by the accused.",
            "legal_issues": "Whether suppression of prior convictions for violent crimes and subsequent threats to key prosecution witnesses justify cancellation of bail under Section 439(2) CrPC.",
            "judgment_reason": "The High Court found that the trial court ignored vital criminal antecedents. Given the credible threat assessment for the primary witnesses, the order granting bail was perverse and undermined the administration of justice.",
            "summary": "Delhi High Court cancelled bail granted to the respondents due to suppression of criminal antecedents and substantiated witness intimidation.",
            "bias_flag": False,
            "parity_argument_used": True,
            "region": "Delhi"
        }
        ),
        "tool_trace": [
            AIMessage(
                content="",
                tool_calls=[{
                    "name": "execute_cypher_query",
                    "args": {
                        "query": """
                            MERGE (c:Case {id: 'DEL-HC-2022-7819'})
                            SET c.title = 'State of NCT of Delhi vs. Manpreet Singh & Ors.',
                                c.date = date('2022-09-14'),
                                c.bail_cancellation_case = true,
                                c.landmark_case = false,
                                c.crime_type = 'Armed Robbery & Criminal Conspiracy',
                                c.facts = 'The incident involved the armed interception of a logistics van transporting cash. Bail was initially granted by the Sessions Court on medical grounds and parity. The State filed for cancellation after ballistic reports confirmed licensed weapon misuse and the complainant reported physical intimidation by the accused.',
                                c.legal_issues = 'Whether suppression of prior convictions for violent crimes and subsequent threats to key prosecution witnesses justify cancellation of bail under Section 439(2) CrPC.',
                                c.judgment_reason = 'The High Court found that the trial court ignored vital criminal antecedents. Given the credible threat assessment for the primary witnesses, the order granting bail was perverse and undermined the administration of justice.',
                                c.summary = 'Delhi High Court cancelled bail granted to the respondents due to suppression of criminal antecedents and substantiated witness intimidation.',
                                c.bias_flag = false,
                                c.parity_argument_used = true,
                                c.region = 'Delhi',
                                c.had_other_accused = true,
                                c.accused_count = 3
                            RETURN c.id AS case_id"""
                    },
                    "id": tool_call_id_1,
                    "type": "tool_call"
                }],
            ),
            ToolMessage(
                content="[{'case_id': 'DEL-HC-2022-7819'}]",
                tool_call_id=tool_call_id_1,
            ),
            AIMessage(
                content="",
                tool_calls=[{
                    "name": "execute_cypher_query",
                    "args": {
                        "query": """
                            MATCH (c:Case {id: 'DEL-HC-2022-7819'})
                            MERGE (court:Court {name: 'Delhi High Court'})
                            MERGE (court)-[:HEARD]->(c)"""
                    },
                    "id": tool_call_id_2,
                    "type": "tool_call"
                }],
            ),
            ToolMessage(
                content="[]",
                tool_call_id=tool_call_id_2,
            ),
            AIMessage(
                content="",
                tool_calls=[{
                    "name": "execute_cypher_query",
                    "args": {
                        "query": """
                            MATCH (c:Case {id: 'DEL-HC-2022-7819'})
                            MERGE (b:Bail {id: 'BAIL-DEL-HC-2022-7819'})
                            SET b.type = 'Regular',
                                b.outcome = 'Rejected',
                                b.outcome_detailed = 'Bail previously granted was revoked due to suppression of prior criminal record and active witness tampering.'
                            MERGE (c)-[:HAS_BAIL]->(b)
                            RETURN b.id AS bail_id"""
                    },
                    "id": tool_call_id_3,
                    "type": "tool_call"
                }],
            ),
            ToolMessage(
                content="[{'bail_id': 'BAIL-DEL-HC-2022-7819'}]",
                tool_call_id=tool_call_id_3,
            ),
            AIMessage(
                content="",
                tool_calls=[{
                    "name": "execute_cypher_query",
                    "args": {
                        "query": """
                            MATCH (c:Case {id: 'DEL-HC-2022-7819'})
                            UNWIND ['Justice Mukta Gupta', 'Justice Anish Dayal'] AS judge_name
                            MERGE (j:Person {name: trim(judge_name)})
                            ON CREATE SET j.type = 'Judge'
                            MERGE (j)-[:JUDGED]->(c)
                            RETURN j.name AS name"""
                    },
                    "id": tool_call_id_4,
                    "type": "tool_call"
                }],
            ),
            ToolMessage(
                content="[{'name': 'Justice Mukta Gupta'}, {'name': 'Justice Anish Dayal'}]",
                tool_call_id=tool_call_id_4,
            ),
            AIMessage(
                content="",
                tool_calls=[{
                    "name": "execute_cypher_query",
                    "args": {
                        "query": """
                            MATCH (c:Case {id: 'DEL-HC-2022-7819'})
                            UNWIND ['Manpreet Singh', 'Harvinder Gill'] AS accused_name
                            MERGE (a:Person {name: trim(accused_name)})
                            ON CREATE SET a.type = 'Accused',
                                          a.prior_cases = 'Yes'
                            MERGE (a)-[:ACCUSED_IN]->(c)
                            RETURN a.name AS name"""
                    },
                    "id": tool_call_id_5,
                    "type": "tool_call"
                }],
            ),
            ToolMessage(
                content="[{'name': 'Manpreet Singh'}, {'name': 'Harvinder Gill'}]",
                tool_call_id=tool_call_id_5,
            ),
            AIMessage(
                content="Case is created with Id: DEL-HC-2022-7819"
            )
        ],
    }
]

for example in examples:
    example["schema"] = schema

example_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(SYSTEM_PROMPT_TEMPLATE),
    HumanMessagePromptTemplate.from_template("Process this dictionary into graph nodes:: \n{input_data}\n"),
    MessagesPlaceholder("tool_trace"),
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    )

final_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(SYSTEM_PROMPT_TEMPLATE),
    few_shot_prompt,
    HumanMessagePromptTemplate.from_template("Process this dictionary into graph nodes:: \n{input_data}\n"),
    ("placeholder", "{agent_scratchpad}"),
])

In [7]:
# Creating LLM instance using Ollama

if LLM_HOST is None or LLM_MODEL is None:
    raise ValueError("OLLAMA_HOST and OLLAMA_MODEL must be set in the environment variables.")

llm = ChatOllama(
    model=LLM_MODEL,
    validate_model_on_init=True,
    temperature=0.5,       # Deterministic Cypher syntax
    num_ctx=16384,         # Plenty of room for schema + few-shot + history
    num_predict=2048,      # Per-turn output budget (prevents mid-query cutoffs)
    )


In [8]:
# Creating a tool calling agent to generate Cypher queries based on user questions and execute them against the Neo4j database

agent = create_tool_calling_agent(llm, [execute_cypher_query], final_prompt)
agent_executor = AgentExecutor(
    agent=agent, 
    tools=[execute_cypher_query], 
    verbose=True,
    handle_parsing_errors=True,
    )

In [9]:
# Importing data
dataframe = pandas.read_csv("../data/Indian-Bail-Judgments new.csv")

## Understanding Data

In [10]:
# Understanding data
dataframe['case_id'] = dataframe['case_id'].astype(str)
dataframe.describe(include='all')

,case_id,case_title,court,date,judge,bail_type,bail_cancellation_case,landmark_case,accused_name,accused_gender,...,bail_outcome,bail_outcome_label_detailed,crime_type,facts,legal_issues,judgment_reason,summary,bias_flag,parity_argument_used,region
count,1200,1200,1200,1200,1200,1200,1200,1200,1200,1200,...,1200,1200,1200,1200,1200,1200,1200,1200,1200,1200
unique,1200,1182,78,884,573,6,2,2,1157,4,...,2,1172,12,1198,1195,1197,1198,2,2,28
top,923,Sumit Kumar vs State Of Delhi,Punjab and Haryana High Court,2024-12-11,Justice Rajesh Bhardwaj,Regular,False,False,Not specified,Male,...,Granted,Regular bail granted after compliance with int...,Narcotics,Ram Niwas Gupta was arrested while attempting ...,Whether preventive detention is valid when the...,The Delhi High Court quashed the detention cit...,Delhi High Court quashed COFEPOSA detention of...,False,False,Uttar Pradesh
freq,1,3,130,32,34,906,1084,1053,11,1071,...,736,18,134,2,2,2,2,1187,859,130


In [11]:
# dataframe.columns

In [12]:
# Understanding data
# dataframe.accused_gender.unique()

## Parsing data

In [13]:
# Parsing data
case_dict = dataframe.to_dict(orient='records')
type(case_dict)

list

In [14]:
agent_executor.invoke({
    "schema": schema,
    "input_data": json.dumps(case_dict[0])
})



> Entering new AgentExecutor chain...

Invoking: `execute_cypher_query` with `{'query': "\n                            MERGE (c:Case {id: '923'})\n                            SET c.title = 'The State Of Maharashtra vs Tukaram Shiva Patil',\n                                c.date = date('1975-08-18'),\n                                c.bail_cancellation_case = true,\n                                c.landmark_case = true,\n                                c.crime_type = 'Attempt to Murder',\n                                c.facts = 'The case involved the murder of Maruti Avaba Patil by seven accused who allegedly used axes and sticks. The prosecution relied on dying declarations. Bail was granted initially on grounds such as incomplete charge-sheet and health condition of an accused. However, the State applied for cancellation citing threats to witnesses and public tension in the village.',\n                                c.legal_issues = 'Whether bail in a serious offence like murde

{'schema': 'Node properties:\nCase {bias_flag: BOOLEAN, parity_argument_used: BOOLEAN, region: STRING, facts: STRING, legal_issues: STRING, judgment_reason: STRING, summary: STRING, bail_cancellation_case: BOOLEAN, landmark_case: BOOLEAN, crime_type: STRING, id: STRING, title: STRING, date: STRING}\nPerson {id: STRING, type: STRING, gender: STRING, prior_cases: STRING, name: STRING}\nBail {outcome_labellled: STRING, id: STRING, type: STRING, outcome: STRING}\nCourt {name: STRING}\nRelationship properties:\n\nThe relationships:\n(:Person)-[:ACCUSED_IN]->(:Case)\n(:Person)-[:JUDGED]->(:Case)\n(:Bail)-[:RELATED_TO]->(:Case)\n(:Court)-[:HEARD]->(:Case)',
 'input_data': '{"case_id": "923", "case_title": "The State Of Maharashtra vs Tukaram Shiva Patil", "court": "Bombay High Court", "date": "1975-08-18", "judge": "Justice Vaidya, Justice J.", "bail_type": "Regular", "bail_cancellation_case": true, "landmark_case": true, "accused_name": "Tukaram Shiva Patil and others", "accused_gender": "Ma

In [15]:
unique_judge_list = []
unique_accused_list = []
case_entity = None
for case in case_dict:
    case_entity = agent_executor.invoke({
        "schema": schema,
        "input_data": json.dumps(case)
    })
    break



> Entering new AgentExecutor chain...

Invoking: `execute_cypher_query` with `{'query': "\n                            MERGE (c:Case {id: '923'})\n                            ON CREATE SET c.title = 'The State Of Maharashtra vs Tukaram Shiva Patil',\n                                          c.date = date('1975-08-18'),\n                                          c.landmark_case = true,\n                                          c.crime_type = 'Attempt to Murder',\n                                          c.facts = 'The case involved the murder of Maruti Avaba Patil by seven accused who allegedly used axes and sticks. The prosecution relied on dying declarations. Bail was granted initially on grounds such as incomplete charge-sheet and health condition of an accused. However, the State applied for cancellation citing threats to witnesses and public tension in the village.',\n                                          c.legal_issues = 'Whether bail in a serious offence like murder can 

In [16]:
print(case_entity)

{'schema': 'Node properties:\nCase {bias_flag: BOOLEAN, parity_argument_used: BOOLEAN, region: STRING, facts: STRING, legal_issues: STRING, judgment_reason: STRING, summary: STRING, bail_cancellation_case: BOOLEAN, landmark_case: BOOLEAN, crime_type: STRING, id: STRING, title: STRING, date: STRING}\nPerson {id: STRING, type: STRING, gender: STRING, prior_cases: STRING, name: STRING}\nBail {outcome_labellled: STRING, id: STRING, type: STRING, outcome: STRING}\nCourt {name: STRING}\nRelationship properties:\n\nThe relationships:\n(:Person)-[:ACCUSED_IN]->(:Case)\n(:Person)-[:JUDGED]->(:Case)\n(:Bail)-[:RELATED_TO]->(:Case)\n(:Court)-[:HEARD]->(:Case)', 'input_data': '{"case_id": "923", "case_title": "The State Of Maharashtra vs Tukaram Shiva Patil", "court": "Bombay High Court", "date": "1975-08-18", "judge": "Justice Vaidya, Justice J.", "bail_type": "Regular", "bail_cancellation_case": true, "landmark_case": true, "accused_name": "Tukaram Shiva Patil and others", "accused_gender": "Mal

In [17]:
close_neo4j_driver()


[Neo4j Cleanup] Closing active session...
[Neo4j Cleanup] session closed successfully.

[Neo4j Cleanup] Closing active connection pool...
[Neo4j Cleanup] Driver shut down successfully.
